# 🚗 Indian ANPR System - Free Cloud Deployment
**Hardware:** Google Cloud (12.7 GB RAM + Free GPU/CPU)  
**Status:** 100% Free, Zero Out-Of-Memory Crashes  
---
### Instructions:
1. Replace `<YOUR_GITHUB_REPO_URL>` below with your GitHub repo URL.
2. Click the **Play (Run)** button on the code cell below.
3. Click the generated **Public HTTPS Link** to open your live ANPR web app!

In [ ]:
# Step 1: Clone your GitHub repository
# Change the URL below to your repository URL:
REPO_URL = "https://github.com/vimal-2320/anpr-system.git"

import os, subprocess, time, re
!rm -rf anpr
!git clone $REPO_URL anpr
%cd anpr

# Step 2: Install required packages (CPU PyTorch, YOLOv8, EasyOCR, Flask)
print("Installing dependencies (takes ~1 minute)...")
!pip install -q -r requirements.txt

# Step 3: Install Cloudflare Tunnel for direct HTTPS access (No password needed)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# Step 4: Start Flask Backend Server in background
print("Starting ANPR Backend Server...")
flask_proc = subprocess.Popen(["python", "app.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

# Step 5: Start Cloudflare Tunnel and display public URL
print("Connecting to Cloudflare Global Edge Tunnel...")
cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

for _ in range(40):
    line = cf_proc.stdout.readline()
    if not line: continue
    m = url_pattern.search(line)
    if m:
        public_url = m.group(0)
        break
    time.sleep(0.5)

print("=" * 60)
if public_url:
    print("🎉 YOUR ANPR APP IS LIVE ONLINE IN THE CLOUD!")
    print(f"🔗 PUBLIC LINK: {public_url}")
    print("=" * 60)
    print("Share this link with anyone! Open it on your phone or laptop.")
else:
    print("Tunnel started. Fallback to localtunnel:")
    !npx -y localtunnel --port 5000
